In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd

spark = SparkSession.builder.appName("Lab4").getOrCreate()

In [0]:
df = spark.read.format("csv").option("header", "true").load("dbfs:/FileStore/shared_uploads/gutjakub@student.agh.edu.pl/online_retail_dataset.csv")

In [0]:
display(df.limit(10))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047,United Kingdom


In [0]:
df.select("*").where(col("Description").isNull()).limit(5).show()

+---------+---------+-----------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+-----------+--------+---------------+---------+----------+--------------+
|   536414|    22139|       null|      56|12/1/2010 11:52|        0|      null|United Kingdom|
|   536545|    21134|       null|       1|12/1/2010 14:32|        0|      null|United Kingdom|
|   536546|    22145|       null|       1|12/1/2010 14:33|        0|      null|United Kingdom|
|   536547|    37509|       null|       1|12/1/2010 14:33|        0|      null|United Kingdom|
|   536549|   85226A|       null|       1|12/1/2010 14:34|        0|      null|United Kingdom|
+---------+---------+-----------+--------+---------------+---------+----------+--------------+



In [0]:
df.select('*').where(col('Description').isNull() ).limit(5).show()
#col("CustomerID").isNotNull() & 
# nie ma brakujacych wartosci dla tylko description jesli jest customerID

+---------+---------+-----------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+-----------+--------+---------------+---------+----------+--------------+
|   536414|    22139|       null|      56|12/1/2010 11:52|        0|      null|United Kingdom|
|   536545|    21134|       null|       1|12/1/2010 14:32|        0|      null|United Kingdom|
|   536546|    22145|       null|       1|12/1/2010 14:33|        0|      null|United Kingdom|
|   536547|    37509|       null|       1|12/1/2010 14:33|        0|      null|United Kingdom|
|   536549|   85226A|       null|       1|12/1/2010 14:34|        0|      null|United Kingdom|
+---------+---------+-----------+--------+---------------+---------+----------+--------------+



In [0]:
#df.select('InvoiceNo').distinct().count()
#df.printSchema()
#fill
df_filled = df.na.fill({'Description':'Brak Opisu'})

In [0]:
display(df_filled.select('*').where(col('Description')=='Brak Opisu').limit(10))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536414,22139,Brak Opisu,56,12/1/2010 11:52,0,null,United Kingdom
536545,21134,Brak Opisu,1,12/1/2010 14:32,0,null,United Kingdom
536546,22145,Brak Opisu,1,12/1/2010 14:33,0,null,United Kingdom
536547,37509,Brak Opisu,1,12/1/2010 14:33,0,null,United Kingdom
536549,85226A,Brak Opisu,1,12/1/2010 14:34,0,null,United Kingdom
536550,85044,Brak Opisu,1,12/1/2010 14:34,0,null,United Kingdom
536552,20950,Brak Opisu,1,12/1/2010 14:34,0,null,United Kingdom
536553,37461,Brak Opisu,3,12/1/2010 14:35,0,null,United Kingdom
536554,84670,Brak Opisu,23,12/1/2010 14:35,0,null,United Kingdom
536589,21777,Brak Opisu,-10,12/1/2010 16:50,0,null,United Kingdom


In [0]:
#explode 
data = [
    ("Order1", ["apple", "banana", "orange"]),
    ("Order2", ["kiwi", "mango"]),
    ("Order3", [])
]
df_explode = spark.createDataFrame(data,["Order_id","Owoce"])
df_explode.show()

+--------+--------------------+
|Order_id|               Owoce|
+--------+--------------------+
|  Order1|[apple, banana, o...|
|  Order2|       [kiwi, mango]|
|  Order3|                  []|
+--------+--------------------+



In [0]:
df_explode.withColumn("Owoc", explode("Owoce")).show()

+--------+--------------------+------+
|Order_id|               Owoce|  Owoc|
+--------+--------------------+------+
|  Order1|[apple, banana, o...| apple|
|  Order1|[apple, banana, o...|banana|
|  Order1|[apple, banana, o...|orange|
|  Order2|       [kiwi, mango]|  kiwi|
|  Order2|       [kiwi, mango]| mango|
+--------+--------------------+------+



In [0]:
df_explode.select("*", posexplode("Owoce").alias("pos", "Owoc")).show()

+--------+--------------------+---+------+
|Order_id|               Owoce|pos|  Owoc|
+--------+--------------------+---+------+
|  Order1|[apple, banana, o...|  0| apple|
|  Order1|[apple, banana, o...|  1|banana|
|  Order1|[apple, banana, o...|  2|orange|
|  Order2|       [kiwi, mango]|  0|  kiwi|
|  Order2|       [kiwi, mango]|  1| mango|
+--------+--------------------+---+------+



In [0]:
#drops
df_drop = df.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))

In [0]:
df_drop.limit(10).show()
df_drop.printSchema()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 08:26:00|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS S

In [0]:
#df_drop.select('*').where(to_date(col('InvoiceDate')) == lit('2011-04-01')).show()
df_drop = df_drop.drop('Country')

In [0]:
df_drop.limit(10).show()

+---------+---------+--------------------+--------+-------------------+---------+----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|
+---------+---------+--------------------+--------+-------------------+---------+----------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 08:26:00|     7.65|     17850|
|   536365|    21730|GLASS STAR FROSTE...|       6|2010-12-01 08:26:00|     4.25|     17850|
|   536366|    22633|HAND WARMER UNION...|       6|2010-12-01 08:28:00

In [0]:
#regex
df_regex = df.withColumn("OnlyLetters", regexp_extract(col("StockCode"), r"([A-Za-z]+)", 1))
df_regex.limit(10).show()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|OnlyLetters|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+-----------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|          A|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|           |
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|          B|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|          G|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|          E|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|    

In [0]:
df_regex = df.withColumn("StockCode", regexp_replace(col("StockCode"),"1","?")) 
df_regex.limit(10).show()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85?23A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    7?053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|     17850|United Kingdom|
|   536365|    2?730|GLASS STAR FROSTE...|       6|12/1/2010 8:26|     4.

In [0]:
#ifnull nullif
#df_drop.select('*').where(col('Description').isNull()).show()
#date '2010-12-01' 
#roznica z ' "
df_drop.withColumn('IfNullDescription', expr("ifnull(Description, 'Brak Opisu') "))\
    .filter( (to_date(col('InvoiceDate')) == lit('2010-12-01')) & ( col("Description").isNull() )) \
    .show()

+---------+---------+-----------+--------+-------------------+---------+----------+-----------------+
|InvoiceNo|StockCode|Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|IfNullDescription|
+---------+---------+-----------+--------+-------------------+---------+----------+-----------------+
|   536414|    22139|       null|      56|2010-12-01 11:52:00|        0|      null|       Brak Opisu|
|   536545|    21134|       null|       1|2010-12-01 14:32:00|        0|      null|       Brak Opisu|
|   536546|    22145|       null|       1|2010-12-01 14:33:00|        0|      null|       Brak Opisu|
|   536547|    37509|       null|       1|2010-12-01 14:33:00|        0|      null|       Brak Opisu|
|   536549|   85226A|       null|       1|2010-12-01 14:34:00|        0|      null|       Brak Opisu|
|   536550|    85044|       null|       1|2010-12-01 14:34:00|        0|      null|       Brak Opisu|
|   536552|    20950|       null|       1|2010-12-01 14:34:00|        0|      null

In [0]:
df.withColumn("Country_temp", expr("nullif(Country,'United Kingdom') ") ).filter(col('Country')=='United Kingdom').show()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|Country_temp|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|        null|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|        null|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|        null|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|        null|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|        null|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7

In [0]:
df_rep = df.replace({'United Kingdom': 'UK'}, subset=['Country'])
df_rep.limit(10).show()

+---------+---------+--------------------+--------+--------------+---------+----------+-------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+--------------------+--------+--------------+---------+----------+-------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|     UK|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|     UK|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|     UK|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|     UK|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|     UK|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|     17850|     UK|
|   536365|    21730|GLASS STAR FROSTE...|       6|12/1/2010 8:26|     4.25|     17850|     UK|
|   536366|    22633|HAND WARMER UNION..

In [0]:
#array contains
df.withColumn("DescriptionWords", split(col("Description"), " "))\
    .filter(array_contains(col("DescriptionWords"),"HAND"))\
    .limit(10).show()

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+--------------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|    DescriptionWords|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+--------------------+
|   536366|    22633|HAND WARMER UNION...|       6| 12/1/2010 8:28|     1.85|     17850|United Kingdom|[HAND, WARMER, UN...|
|   536366|    22632|HAND WARMER RED P...|       6| 12/1/2010 8:28|     1.85|     17850|United Kingdom|[HAND, WARMER, RE...|
|   536372|    22632|HAND WARMER RED P...|       6| 12/1/2010 9:01|     1.85|     17850|United Kingdom|[HAND, WARMER, RE...|
|   536372|    22633|HAND WARMER UNION...|       6| 12/1/2010 9:01|     1.85|     17850|United Kingdom|[HAND, WARMER, UN...|
|   536377|    22632|HAND WARMER RED P...|       6| 12/1/2010 9:34|     1.85|     17850|United Kingdom|[HAND, WARMER, RE...|


In [0]:
df.select(avg('UnitPrice')).alias('SredniaCen').show()

+----------------+
|  avg(UnitPrice)|
+----------------+
|4.61111362608971|
+----------------+



In [0]:
#Liczba produktow
df.select('StockCode').distinct().count()

Out[140]: 4070

In [0]:
#MAXDATE
df_drop.agg(max("InvoiceDate").alias("MaxDate")).show()


+-------------------+
|            MaxDate|
+-------------------+
|2011-12-09 12:50:00|
+-------------------+



In [0]:
from pyspark.sql.types import DoubleType, StringType
'''
def adjust_pound_price(price,country):
    if price is None or country is None:
        return None
    if country.strip().lower() == 'united kingdom':
        return round(price * 0.85, 2)
    return price
'''
def adjust_price(price):
    if price > 10:
        return price * 0.95  

adjust_price_udf = udf(adjust_price, DoubleType())

display(df.withColumn("NewPrice", adjust_price_udf(col("UnitPrice"))).limit(10))




InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,NewPrice
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom,null
536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom,null
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom,null
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom,null
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom,null
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850,United Kingdom,null
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850,United Kingdom,null
536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850,United Kingdom,null
536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850,United Kingdom,null
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047,United Kingdom,null


In [0]:

@pandas_udf(StringType())
def country_code_udf(country_series: pd.Series) -> pd.Series:
    return country_series.str[:2].str.upper()
df.withColumn("CountryCode", country_code_udf(col("Country"))).show()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|CountryCode|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+-----------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|         UN|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|         UN|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|         UN|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|         UN|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|         UN|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|    